# 10b Seed and Refine the Concept Catalog

Start from the nine TODO seed concepts. This notebook can ask the configured LLM for child-concept hypotheses, but generation is off by default. Review proposed definitions before adding them to the catalog.

Concept-generation examples come only from PMIDs linked exclusively to development pairs, preventing shared papers or confirmation-pair abstracts from entering the prompt. SI values and forecast outcomes are never loaded here.

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "amr").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "amr").exists():
    raise RuntimeError(f"Could not find repository root from {Path.cwd()}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from amr.pipeline_config import cfg, cfg_path, load_pipeline_config
from amr.pipelines.concepts import build_si_concept_paths, load_concept_corpus, corpus_preflight
from amr.si.concepts import (
    SEED_CONCEPTS, concept_catalog_hash, estimate_classification_cost,
    classify_concept_abstracts, build_yearly_concept_signals,
    rolling_origin_concept_forecasts, summarise_incremental_forecast_gains,
    propose_concept_children,
)

PIPELINE_CONFIG = load_pipeline_config(project_root=PROJECT_ROOT)
CONCEPT_PATHS = build_si_concept_paths(PROJECT_ROOT)
MODEL_NAME = str(cfg(PIPELINE_CONFIG, "llm.model", "gpt-5-nano"))

corpus = pd.read_parquet(CONCEPT_PATHS["corpus"])
abstract_table = pd.read_parquet(CONCEPT_PATHS["abstracts"])
catalog_path = CONCEPT_PATHS["catalog"]
if catalog_path.exists():
    catalog = json.loads(catalog_path.read_text())
else:
    catalog = SEED_CONCEPTS.copy()
print("Catalog version:", concept_catalog_hash(catalog), "concepts:", len(catalog))


## Select development-only examples

In [ ]:
PARENT_ID = "surveillance"
development_pmids = set(corpus.loc[corpus["split"].eq("development"), "pmid"].astype(str))
confirmation_pmids = set(corpus.loc[corpus["split"].eq("confirmation"), "pmid"].astype(str))
exclusive_development_pmids = development_pmids - confirmation_pmids
examples_df = pd.DataFrame(columns=["pmid", "title", "abstract"])
if CONCEPT_PATHS["scores"].exists():
    saved_scores = pd.read_parquet(CONCEPT_PATHS["scores"])
    parent_scores = saved_scores.loc[
        saved_scores["concept_id"].eq(PARENT_ID) & saved_scores["status"].eq("ok")
        & saved_scores["pmid"].astype(str).isin(exclusive_development_pmids)
    ].drop_duplicates("pmid", keep="last")
    parent_scores = parent_scores.loc[parent_scores["probability"].ge(0.5)]
    examples_df = (
        parent_scores[["pmid"]].merge(abstract_table, on="pmid", how="inner", validate="one_to_one")
        .loc[lambda frame: frame["abstract_available"]]
        .sample(frac=1, random_state=42)
        .head(30)
    )
examples = examples_df[["title", "abstract"]].to_dict("records")
assert not set(examples_df["pmid"].astype(str)) & confirmation_pmids
print(f"Development-only examples for {PARENT_ID}: {len(examples):,}")
if not examples:
    print("Run notebook 10c on the current parent concept first; refinement needs scored parent examples.")


## Optional child-concept proposal

In [ ]:
RUN_CONCEPT_PROPOSAL = False
PARENT_ID = "surveillance"
if RUN_CONCEPT_PROPOSAL:
    if not examples:
        raise RuntimeError("No development-only scored examples for the parent; classify the current catalog first.")
    from amr.llm import OpenAIConfig, OpenAILanguageModel
    llm = OpenAILanguageModel(OpenAIConfig(model=MODEL_NAME, temperature=0, max_tokens=2400))
    parent = next(concept for concept in catalog if concept["concept_id"] == PARENT_ID)
    proposal = propose_concept_children(parent, examples, llm=llm, model_name=MODEL_NAME)
    proposal_path = CONCEPT_PATHS["directory"] / f"proposals_{PARENT_ID}.json"
    proposal_path.write_text(json.dumps(proposal, indent=2, ensure_ascii=False))
    print("Review proposal before editing catalog:", proposal_path)
else:
    print("LLM proposal disabled. Set RUN_CONCEPT_PROPOSAL=True to request hypotheses.")


## Save the reviewed catalog

In [ ]:
# Add reviewed child dictionaries to this catalog. Give each a stable concept_id,
# parent_id, definition, inclusion_criteria, and exclusion_criteria.
catalog_version = concept_catalog_hash(catalog)
CONCEPT_PATHS["catalog_versions"].mkdir(parents=True, exist_ok=True)
versioned_catalog_path = CONCEPT_PATHS["catalog_versions"] / f"concept_catalog_{catalog_version}.json"
versioned_catalog_path.write_text(json.dumps(catalog, indent=2, ensure_ascii=False))
CONCEPT_PATHS["catalog"].write_text(json.dumps(catalog, indent=2, ensure_ascii=False))
print("Saved catalog:", CONCEPT_PATHS["catalog"])
print("Saved immutable version:", versioned_catalog_path)
print("Catalog version:", catalog_version)
